In [1]:
import os
from dotenv import load_dotenv
load_dotenv()  # loads LOCAL_OPENAI_API_KEY from the project-root .env (never hardcode the key)

"""
Simple connectivity test for the new university LLM endpoint (llmlite.vse.cz).
Run this yourself with: python test_llmlite_endpoint.py
Requires: pip install openai
"""
import socket
import urllib.request
import urllib.error

HOST = "llmlite.vse.cz"
PORT = 443
API_KEY = os.environ["LOCAL_OPENAI_API_KEY"]
BASE_URL = f"https://{HOST}/v1"  # LiteLLM proxies are OpenAI-compatible at /v1

# --- Step 1: raw TCP connectivity ---
print(f"[1] TCP connect to {HOST}:{PORT} ...")
try:
    with socket.create_connection((HOST, PORT), timeout=8) as s:
        print("    OK - TCP connection succeeded")
except OSError as e:
    print(f"    FAILED: {e}")
    raise SystemExit("Stopping - no TCP connectivity at all.")

# --- Step 2: hit /v1/models with plain urllib ---
print(f"[2] GET {BASE_URL}/models ...")
req = urllib.request.Request(f"{BASE_URL}/models", headers={"Authorization": f"Bearer {API_KEY}"})
try:
    with urllib.request.urlopen(req, timeout=15) as resp:
        body = resp.read().decode("utf-8", errors="replace")
        print(f"    HTTP {resp.status}")
        print(f"    Body (first 500 chars): {body[:500]}")
except urllib.error.HTTPError as e:
    print(f"    HTTP ERROR {e.code}: {e.reason}")
    print(f"    Body: {e.read().decode('utf-8', errors='replace')[:500]}")
except urllib.error.URLError as e:
    print(f"    URL ERROR: {e.reason}")

# --- Step 3: minimal chat completion via openai SDK ---
print("[3] Minimal chat completion via openai SDK ...")
try:
    from openai import OpenAI
except ImportError:
    print("    openai package not installed - run: pip install openai")
    raise SystemExit(0)

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
try:
    resp = client.chat.completions.create(
        model="gpt-4o-mini",  # replace with a real model name once step 2 lists them
        messages=[{"role": "user", "content": "Reply with the single word: OK"}],
        max_tokens=10,
    )
    print(f"    OK - model replied: {resp.choices[0].message.content!r}")
except Exception as e:
    print(f"    FAILED: {type(e).__name__}: {e}")

[1] TCP connect to llmlite.vse.cz:443 ...


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



    FAILED: [Errno -2] Name or service not known
Traceback (most recent call last):
  File "/tmp/ipykernel_2933/582046938.py", line 18, in <cell line: 0>
    with socket.create_connection((HOST, PORT), timeout=8) as s:
         ~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/socket.py", line 844, in create_connection
    for res in getaddrinfo(host, port, 0, SOCK_STREAM):
               ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/socket.py", line 983, in getaddrinfo
    for res in _socket.getaddrinfo(host, port, family, type, proto, flags):
               ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
socket.gaierror: [Errno -2] Name or service not known

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
    ~

TypeError: object of type 'NoneType' has no len()